# Lao OCR training server on Colab GPU

Runs the training server on this Colab GPU and opens a Cloudflare tunnel to it.
The web UI runs on your own computer (`webapp/ui`, `npm run dev`). Manual: `webapp/README.md`.

1. **Runtime → Change runtime type → GPU**, then run the cells in order.
2. Copy the printed **URL** and **TOKEN** into the UI's *Settings* page.

How data and results are kept:

* **Code** is cloned from GitHub (it must include `webapp/`).
* **Data** (`data/` inside your zip on Drive) is extracted to Colab's local disk, which is much faster than Drive for training.
* **Results** (job logs and the model folders that web jobs write) are copied back to `RESULTS_DIR` on Drive every 2 minutes
  and restored from there at the next session. Checkpoints are never only on the Colab disk for more than 2 minutes.

In [ ]:
!nvidia-smi --query-gpu=name,memory.total,driver_version --format=csv
!df -h /content | tail -1

## 1. Settings

Edit these if your files are elsewhere. Mount the Google account that owns the zip.

In [ ]:
# Zip with the toolkit and its data/ folder (code from the zip is not used; data/ is)
ZIP_PATH = '/content/drive/MyDrive/Niigata_University/Master/2026/Compared/Update_50k_iters/tesstrain-gpu-20260714.zip'

# Only extract these top-level folders of data/ (e.g. ['lao-ground-truth', 'LaoCRNN-I64K5C16']).
# Empty list = extract all of data/. Use it if the local disk is too small.
DATASETS = []

# Where results are kept on Drive between sessions (also holds the UI token)
RESULTS_DIR = '/content/drive/MyDrive/Niigata_University/Master/2026/Compared/webui_runs'

WORK = '/content/work'                 # local disk: data/ is extracted to WORK/data
REPO_URL = 'https://github.com/Monaliza14xx/Lao_OCR_Toolkit.git'
BRANCH = 'main'
REPO = '/content/Lao_OCR_Toolkit'
SYNC_EVERY_SEC = 120

## 2. Drive, code and dependencies

In [ ]:
import os
from google.colab import drive
drive.mount('/content/drive')

if not os.path.exists(ZIP_PATH):
    parent = os.path.dirname(ZIP_PATH)
    print('Zip not found:', ZIP_PATH)
    print('Files in', parent, ':', sorted(os.listdir(parent)) if os.path.isdir(parent) else 'folder does not exist')
    raise FileNotFoundError(ZIP_PATH)
print(f'Zip: {os.path.getsize(ZIP_PATH) / 1e9:.1f} GB')
os.makedirs(RESULTS_DIR, exist_ok=True)

if os.path.isdir(REPO):
    !git -C {REPO} fetch -q origin {BRANCH} && git -C {REPO} checkout -q {BRANCH} && git -C {REPO} pull -q
else:
    !git clone -q -b {BRANCH} {REPO_URL} {REPO}
!git -C {REPO} log -1 --oneline
assert os.path.isdir(f'{REPO}/webapp/server'), 'The cloned code has no webapp/: push it to GitHub first'

In [ ]:
# Colab already ships CUDA PyTorch; this installs the rest.
!pip install -q -r {REPO}/requirements.txt -r {REPO}/webapp/server/requirements.txt
!sudo apt-get -qq install -y libraqm0 unzip > /dev/null
if not os.path.exists('/usr/local/bin/cloudflared'):
    !wget -q -O /usr/local/bin/cloudflared https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64
    !chmod +x /usr/local/bin/cloudflared
!cloudflared --version

## 3. Extract data and restore earlier results

The first run in a session reads the whole zip from Drive (about 10–20 minutes for 20 GB).
Re-running this cell in the same session skips the extraction.

In [ ]:
import sys, time
sys.path.insert(0, f'{REPO}/webapp')
import importlib, colab_setup
importlib.reload(colab_setup)

t0 = time.time()
DATA_DIR = str(colab_setup.extract_data(ZIP_PATH, WORK, DATASETS))
print(f'done in {(time.time() - t0) / 60:.1f} min')
print('restored files from Drive:', colab_setup.restore_from_drive(RESULTS_DIR, DATA_DIR))
print('data/ contains:', sorted(os.listdir(DATA_DIR))[:30])
!du -sh {DATA_DIR}

## 4. Start the server, the tunnel and the Drive sync

In [ ]:
import re, secrets, subprocess, threading, urllib.request

token_file = f'{RESULTS_DIR}/.webui_token'
if os.path.exists(token_file):
    TOKEN = open(token_file).read().strip()
else:
    TOKEN = secrets.token_urlsafe(24)
    with open(token_file, 'w') as f:
        f.write(TOKEN)

for name in ('server', 'tunnel'):
    p = globals().get(f'{name}_proc')
    if p and p.poll() is None:
        p.terminate()

server_proc = subprocess.Popen(
    ['python', '-m', 'uvicorn', 'webapp.server.app:create_app', '--factory',
     '--host', '127.0.0.1', '--port', '8000'],
    cwd=REPO, env={**os.environ, 'LAOOCR_TOKEN': TOKEN, 'LAOOCR_DATA': DATA_DIR},
    stdout=open('/content/server.log', 'w'), stderr=subprocess.STDOUT)

def health():
    req = urllib.request.Request('http://127.0.0.1:8000/health', headers={'Authorization': f'Bearer {TOKEN}'})
    return urllib.request.urlopen(req, timeout=30).read().decode()

for _ in range(60):
    try:
        print('server:', health()); break
    except Exception:
        if server_proc.poll() is not None:
            raise RuntimeError(open('/content/server.log').read())
        time.sleep(1)

tunnel_proc = subprocess.Popen(
    ['cloudflared', 'tunnel', '--no-autoupdate', '--url', 'http://127.0.0.1:8000'],
    stdout=subprocess.DEVNULL, stderr=open('/content/tunnel.log', 'w'))
URL = None
for _ in range(60):
    m = re.search(r'https://[a-z0-9-]+\.trycloudflare\.com', open('/content/tunnel.log').read())
    if m:
        URL = m.group(0); break
    time.sleep(1)

# Copy results to Drive in the background
if globals().get('sync_stop'):
    globals()['sync_stop'].set()
sync_stop = threading.Event()
sync_state = {'last': None, 'files': 0, 'error': None}
def _sync_loop(stop=sync_stop):
    while not stop.wait(SYNC_EVERY_SEC):
        try:
            sync_state['files'] += colab_setup.sync_to_drive(DATA_DIR, RESULTS_DIR)
            sync_state['last'], sync_state['error'] = time.strftime('%H:%M:%S'), None
        except Exception as e:
            sync_state['error'] = repr(e)
threading.Thread(target=_sync_loop, daemon=True).start()

print('\n' + '=' * 60)
print('Paste these into the web UI → Settings')
print('URL:  ', URL or 'not found, see /content/tunnel.log')
print('TOKEN:', TOKEN)
print('=' * 60)

## 5. Keep alive

Leave this running while you train. It prints server and sync status every minute.
Colab can still disconnect an idle browser tab; the job then shows as *lost*, but checkpoints synced to Drive are kept.

In [ ]:
import json
try:
    while True:
        if server_proc.poll() is not None:
            print('server exited:'); print(open('/content/server.log').read()[-3000:]); break
        h = json.loads(health())
        print(time.strftime('%H:%M:%S'), h['gpu'].get('name', 'no GPU'), 'free', h['gpu'].get('memory_free_gb'), 'GB',
              '| job:', h['running_job'] or 'idle',
              '| synced:', sync_state['last'] or '-', sync_state['error'] or '')
        time.sleep(60)
except KeyboardInterrupt:
    print('stopped watching (server keeps running)')

## 6. Stop everything (copies results to Drive first)

In [ ]:
print('final sync, files copied:', colab_setup.sync_to_drive(DATA_DIR, RESULTS_DIR))
sync_stop.set()
for p in (globals().get('tunnel_proc'), globals().get('server_proc')):
    if p and p.poll() is None:
        p.terminate()
print('stopped')